# Обнаружение сетевых атак с использованием методов машинного обучения

## 1. Исследовательский анализ данных (EDA)

Датасет: **CICIDS2017** — Canadian Institute for Cybersecurity
- 2.8 млн записей сетевого трафика
- 15 типов атак + нормальный трафик
- 78 признаков

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = '../data'
MODELS_DIR = '../app/models'

## 1.1 Загрузка и очистка данных

In [ ]:
from app.ml.preprocessing import load_dataset, clean_data, map_labels, select_features

df = load_dataset(DATA_DIR)
print(f'Размер датасета: {df.shape}')
print(f'\nПервые 5 строк:')
df.head()

In [ ]:
df = clean_data(df)
df = map_labels(df)

## 1.2 Распределение классов

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Бинарное распределение
binary_counts = df['is_attack'].value_counts()
labels_binary = ['Нормальный трафик', 'Атака']
colors_binary = ['#2ecc71', '#e74c3c']
axes[0].pie(binary_counts, labels=labels_binary, colors=colors_binary,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 14})
axes[0].set_title('Нормальный трафик vs Атаки', fontsize=16)

# Распределение по типам атак
attack_counts = df[df['is_attack'] == 1]['attack_category'].value_counts()
attack_counts.plot(kind='barh', ax=axes[1], color='#e74c3c', edgecolor='black')
axes[1].set_title('Распределение типов атак', fontsize=16)
axes[1].set_xlabel('Количество записей')

plt.tight_layout()
plt.savefig('../app/models/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.3 Корреляционная матрица признаков

In [ ]:
features = select_features(df)
# Берём подмножество признаков для читаемости
top_features = features[:20]

corr = df[top_features].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Корреляционная матрица (топ-20 признаков)', fontsize=16)
plt.tight_layout()
plt.savefig('../app/models/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 Статистика признаков

In [ ]:
print('Статистика по основным признакам:')
df[features[:10]].describe().round(2)

---
## 2. Обучение моделей

### 2.1 Подготовка данных для обучения

In [ ]:
from app.ml.preprocessing import prepare_data

data = prepare_data(DATA_DIR, MODELS_DIR)

print(f"Признаков: {data['n_features']}")
print(f"Классов: {data['n_classes']}")
print(f"Классы: {data['class_names']}")
print(f"Train: {len(data['X_train'])}, Test: {len(data['X_test'])}")
print(f"Нормальный трафик (для автоэнкодера): {len(data['X_train_normal'])}")

### 2.2 Обучение автоэнкодера

In [ ]:
from app.ml.autoencoder import AnomalyDetector

anomaly_detector = AnomalyDetector(data['n_features'])
ae_history = anomaly_detector.train(data['X_train_normal'], epochs=50, batch_size=256)

# График обучения
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ae_history['train_loss'], label='Train Loss', linewidth=2)
ax.plot(ae_history['val_loss'], label='Validation Loss', linewidth=2)
ax.set_xlabel('Эпоха')
ax.set_ylabel('MSE Loss')
ax.set_title('Обучение автоэнкодера', fontsize=16)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('../app/models/autoencoder_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Подбор порога и оценка
anomaly_detector.find_threshold(data['X_test'], data['y_test_binary'])
ae_metrics = anomaly_detector.evaluate(data['X_test'], data['y_test_binary'])
anomaly_detector.save(f'{MODELS_DIR}/autoencoder.pt')

In [ ]:
# Распределение ошибок реконструкции
scores = anomaly_detector.predict_scores(data['X_test'])
normal_scores = scores[data['y_test_binary'] == 0]
attack_scores = scores[data['y_test_binary'] == 1]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(normal_scores, bins=100, alpha=0.7, label='Нормальный трафик',
        color='#2ecc71', density=True)
ax.hist(attack_scores, bins=100, alpha=0.7, label='Атаки',
        color='#e74c3c', density=True)
ax.axvline(anomaly_detector.threshold, color='black', linestyle='--',
           linewidth=2, label=f'Порог = {anomaly_detector.threshold:.4f}')
ax.set_xlabel('Ошибка реконструкции (MSE)')
ax.set_ylabel('Плотность')
ax.set_title('Распределение ошибок реконструкции автоэнкодера', fontsize=16)
ax.legend(fontsize=12)
ax.set_xlim(0, np.percentile(scores, 99))
plt.tight_layout()
plt.savefig('../app/models/reconstruction_error_dist.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Обучение классификаторов

In [ ]:
from app.ml.classifiers import (
    create_classifiers, train_classifier,
    evaluate_classifier, save_classifier,
)

classifiers = create_classifiers(data['n_features'], data['n_classes'])
results = {}

for name, clf in classifiers.items():
    train_classifier(name, clf, data['X_train'], data['y_train_multi'])
    result = evaluate_classifier(
        name, clf, data['X_test'], data['y_test_multi'], data['class_names']
    )
    save_classifier(name, clf, MODELS_DIR)
    results[name] = result

### 2.4 Сравнение моделей

In [ ]:
# Таблица метрик
metrics_df = pd.DataFrame({
    name: r['metrics'] for name, r in results.items()
}).T

print('Сравнение классификаторов:')
metrics_df.round(4)

In [ ]:
# Bar chart сравнения
fig, ax = plt.subplots(figsize=(12, 6))
metrics_to_plot = ['accuracy', 'f1_weighted', 'f1_macro']
x = np.arange(len(results))
width = 0.25

for i, metric in enumerate(metrics_to_plot):
    values = [results[name]['metrics'][metric] for name in results]
    bars = ax.bar(x + i * width, values, width, label=metric.replace('_', ' ').title())

ax.set_xlabel('Модель')
ax.set_ylabel('Значение метрики')
ax.set_title('Сравнение классификаторов', fontsize=16)
ax.set_xticks(x + width)
ax.set_xticklabels(results.keys())
ax.legend(fontsize=12)
ax.set_ylim(0.9, 1.005)
plt.tight_layout()
plt.savefig('../app/models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix для лучшей модели (XGBoost)
best_name = max(results, key=lambda n: results[n]['metrics']['f1_weighted'])
print(f'Лучшая модель: {best_name}')

cm = np.array(results[best_name]['confusion_matrix'])

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data['class_names'],
            yticklabels=data['class_names'], ax=ax)
ax.set_xlabel('Предсказание', fontsize=14)
ax.set_ylabel('Истинное значение', fontsize=14)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=16)
plt.tight_layout()
plt.savefig('../app/models/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Итоги

### Двухуровневая система:
1. **Автоэнкодер** — обнаруживает аномалии (включая zero-day атаки)
2. **Классификатор** — определяет тип атаки

Все модели обучены и сохранены. Готовы к использованию через API.